# 13 · Modelo panel — Regularizado (Ridge / Lasso)

Regresion lineal regularizada sobre el panel completo (one-hot de municipio + features), como contraste lineal/interpretable frente a los GB.

- `Ridge` (L2) y `Lasso` (L1) con alpha elegido por CV sobre el entrenamiento
- Caracteristicas numericas escaladas (StandardScaler); one-hot sin escalar
- Mismo split temporal y prediccion recursiva que 11/12
- Resultados: `results/13_modelo_panel_regularizado_metrics.csv`


In [1]:
import polars as pl
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", palette="muted")
DATA = Path("data")
RESULTS = Path("results")
RESULTS.mkdir(exist_ok=True)

# --- Datos
abast = pl.read_csv(DATA / "abastecimiento_urbano_baleares.csv", infer_schema_length=None).select(["cod_municipio", "anio", "consumo_hm3"])
presion = pl.read_csv(DATA / "presion_humana.csv", infer_schema_length=None)
ocup = pl.read_csv(DATA / "ocupacion_turistica.csv", infer_schema_length=None)
lluvia = pl.read_csv(DATA / "lluvia_masa_subterranea.csv", infer_schema_length=None)
mma = pl.read_csv(DATA / "municipio_masa_subterranea.csv", infer_schema_length=None)
mun = pl.read_csv(DATA / "municipio.csv", infer_schema_length=None).select(["cod_municipio", "cod_provincia", "nombre_municipio"])

# Isla IPH: 071 (Formentera) y 072 (Eivissa) comparten serie NUTS
isla_map = pl.DataFrame({
    "cod_provincia": [71, 72, 73, 74],
    "isla": ["Eivissa i Formentera", "Eivissa i Formentera", "Mallorca", "Menorca"],
})

# --- Features
iph = (presion.group_by(["nombre_isla", "anio"])
       .agg(iph_media=pl.col("iph").mean(), iph_max=pl.col("iph").max()))
ocup_m = (ocup.group_by(["cod_municipio_ine", "anio"])
          .agg(ocupacion_media=pl.col("ocupacion_plazas_pct").mean()))
ll_m = (lluvia.group_by(["cod_masa", "anio"])
        .agg(lluvia_anual_mm=pl.col("precipitacion_mm").sum())
        .join(mma, on="cod_masa")
        .group_by(["cod_municipio", "anio"])
        .agg(lluvia_anual_mm=pl.col("lluvia_anual_mm").mean()))

panel = (
    abast
    .join(mun, on="cod_municipio", how="left")
    .join(isla_map, on="cod_provincia", how="left")
    .join(iph, left_on=["isla", "anio"], right_on=["nombre_isla", "anio"], how="left")
    .join(ocup_m, left_on=["cod_municipio", "anio"], right_on=["cod_municipio_ine", "anio"], how="left")
    .join(ll_m, on=["cod_municipio", "anio"], how="left")
    .select([
        "cod_municipio", "nombre_municipio", "isla", "anio", "consumo_hm3",
        "iph_media", "iph_max", "ocupacion_media", "lluvia_anual_mm",
    ])
    .with_columns(
        pl.col("ocupacion_media").fill_null(0.0),
        pl.col("lluvia_anual_mm").fill_null(pl.col("lluvia_anual_mm").mean()),
    )
)

TEST_START = 2022
FEATURES = ["anio", "iph_media", "iph_max", "ocupacion_media", "lluvia_anual_mm", "lag1"]

panel = (
    panel
    .filter(pl.col("anio") >= 2015)
    .sort(["cod_municipio", "anio"])
    .with_columns(lag1=pl.col("consumo_hm3").shift(1).over("cod_municipio"))
)
print("panel:", panel.shape)
panel.head(8)


panel: (670, 10)


cod_municipio,nombre_municipio,isla,anio,consumo_hm3,iph_media,iph_max,ocupacion_media,lluvia_anual_mm,lag1
i64,str,str,i64,f64,f64,i64,f64,f64,f64
7001,"""Alaró""","""Mallorca""",2015,0.255,1.0826e6,1388045,0.0,408.1,null
7001,"""Alaró""","""Mallorca""",2016,0.278,1.1121e6,1415443,0.0,384.1,0.255
7001,"""Alaró""","""Mallorca""",2017,0.277,1.13149e6,1434094,0.0,496.2,0.278
7001,"""Alaró""","""Mallorca""",2018,0.263,1.1359e6,1414410,0.0,762.0,0.277
7001,"""Alaró""","""Mallorca""",2019,0.279,1.1458e6,1424487,0.0,457.0,0.263
7001,"""Alaró""","""Mallorca""",2020,0.277,981758.416667,1097115,0.0,584.7,0.279
7001,"""Alaró""","""Mallorca""",2021,0.289,1.0554e6,1261233,0.0,620.6,0.277
7001,"""Alaró""","""Mallorca""",2022,0.294,1149634.5,1422758,0.0,455.8,0.289


In [2]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def metricas(test, pred):
    test, pred = np.asarray(test, float), np.asarray(pred, float)
    return dict(
        mae=float(mean_absolute_error(test, pred)),
        mape=float(np.mean(np.abs((test - pred) / test)) * 100),
        rmse=float(mean_squared_error(test, pred) ** 0.5),
        r2=float(r2_score(test, pred)),
    )

def particiones(g):
    g = g.sort("anio")
    train = g.filter(pl.col("anio") < TEST_START)
    test = g.filter(pl.col("anio") >= TEST_START)
    return train, test


In [3]:
from sklearn.linear_model import Ridge, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold

pdf = panel.to_pandas()
dummies = pd.get_dummies(pdf["cod_municipio"], prefix="mun", dtype=float)
X_all = pd.concat([pdf[FEATURES], dummies], axis=1)
X_all["cod_municipio"] = pdf["cod_municipio"].values
X_all["consumo_hm3"] = pdf["consumo_hm3"].values
numeric = ["anio", "iph_media", "iph_max", "ocupacion_media", "lluvia_anual_mm", "lag1"]
feature_cols = [c for c in X_all.columns if c not in ("cod_municipio", "consumo_hm3")]

X_all_scaled = X_all.copy()
scaler = StandardScaler().fit(X_all[numeric])
X_all_scaled[numeric] = scaler.transform(X_all[numeric])
X_all_scaled["anio_raw"] = X_all["anio"].values  # anio sin escalar, para los filtros temporales
print("features:", len(feature_cols))


features: 73


In [4]:
train = X_all_scaled[X_all_scaled["anio_raw"] < TEST_START].dropna(subset=["lag1"])
y_train = train["consumo_hm3"].values

alphas = [0.01, 0.1, 1.0, 10.0, 100.0]
kf = KFold(n_splits=5, shuffle=True, random_state=42)

def mejor_alpha(Cls):
    scores = {}
    for a in alphas:
        errs = []
        for i_tr, i_va in kf.split(train):
            m = Cls(alpha=a, random_state=42)
            m.fit(train.iloc[i_tr][feature_cols], y_train[i_tr])
            p = m.predict(train.iloc[i_va][feature_cols])
            errs.append(np.mean(np.abs((y_train[i_va] - p) / y_train[i_va])) * 100)
        scores[a] = np.mean(errs)
    return min(scores, key=scores.get), scores

best_ridge, s_ridge = mejor_alpha(Ridge)
best_lasso, s_lasso = mejor_alpha(Lasso)
print("Ridge alphas (MAPE CV):", {k: round(v, 1) for k, v in s_ridge.items()}, "-> mejor:", best_ridge)
print("Lasso alphas (MAPE CV):", {k: round(v, 1) for k, v in s_lasso.items()}, "-> mejor:", best_lasso)

ridge = Ridge(alpha=best_ridge).fit(train[feature_cols], y_train)
lasso = Lasso(alpha=best_lasso, random_state=42).fit(train[feature_cols], y_train)


Ridge alphas (MAPE CV): {0.01: np.float64(30.5), 0.1: np.float64(32.6), 1.0: np.float64(25.3), 10.0: np.float64(19.9), 100.0: np.float64(107.3)} -> mejor: 10.0
Lasso alphas (MAPE CV): {0.01: np.float64(15.8), 0.1: np.float64(29.6), 1.0: np.float64(228.7), 10.0: np.float64(823.0), 100.0: np.float64(823.0)} -> mejor: 0.01


In [5]:
i_lag1 = numeric.index("lag1")
mu_lag1, sd_lag1 = scaler.mean_[i_lag1], scaler.scale_[i_lag1]

def predict_recursivo_lin(model, train_rows, test_rows):
    last = float(train_rows["consumo_hm3"].iloc[-1])
    preds = []
    for _, row in test_rows.iterrows():
        x = {c: row[c] for c in feature_cols}
        x["lag1"] = (last - mu_lag1) / sd_lag1  # lag1 va escalado (StandardScaler)
        p = max(float(model.predict([list(x[f] for f in feature_cols)])[0]), 0.0)
        preds.append(p)
        last = p
    return preds

filas = []
for nombre, model in [("ridge", ridge), ("lasso", lasso)]:
    for cod_t, g in panel.partition_by("cod_municipio", as_dict=True).items():
        cod = int(cod_t[0])
        te = g.filter(pl.col("anio") >= TEST_START).sort("anio")
        tr_p = X_all_scaled[(X_all_scaled["cod_municipio"] == cod) & (X_all_scaled["anio_raw"] < TEST_START) & (X_all_scaled["lag1"].notna())]
        te_p = X_all_scaled[(X_all_scaled["cod_municipio"] == cod) & (X_all_scaled["anio_raw"] >= TEST_START)]
        pred = predict_recursivo_lin(model, tr_p, te_p)
        filas.append({"modelo": nombre, "cod_municipio": cod,
                      **metricas(te["consumo_hm3"].to_list(), pred)})

res = pd.DataFrame(filas)
res.to_csv(RESULTS / "13_modelo_panel_regularizado_metrics.csv", index=False)
print(res.groupby("modelo")[["mae", "mape", "rmse", "r2"]].mean().round(3))


          mae    mape   rmse       r2
modelo                               
lasso   0.235  29.764  0.252 -655.564
ridge   0.217  13.576  0.235  -69.942


In [6]:
# ── Comparacion con los demas modelos
base = pd.read_csv(RESULTS / "10_baseline_metrics.csv")
naive = base[base["modelo"] == "naive"].set_index("cod_municipio")["mape"].sort_index()
ridge_m = res[res["modelo"] == "ridge"].set_index("cod_municipio")["mape"].sort_index()
print("municipios donde ridge mejora al naive:", (ridge_m < naive).sum(), "de", naive.shape[0])


municipios donde ridge mejora al naive: 30 de 67


**Conclusiones**

- Ridge/Lasso ofrecen un contraste lineal interpretable: si quedan lejos de los GB, la relacion consumo-features no es lineal (o el one-hot no basta).
- Lasso ademas permite inspeccionar que features (o municipios) tienen coeficiente no nulo.
